# AI_SUMMARIZE — Wikipedia Article Summarisation

## Use Case Overview

`AI_SUMMARIZE` condenses long-form text into a concise summary. It's designed to handle substantial bodies of text — technical documentation, support tickets, research papers, meeting transcripts — and extract the key points.

**SA use case:** Automatically generate executive summaries of documents, incident reports, or long-form content stored in Snowflake tables — without any external tools or Python code.

**Dataset:** 5 Wikipedia-style articles on technology topics (300–420 words each), stored as text rows in Snowflake.

**What we'll demonstrate:**
1. Basic AI_SUMMARIZE on a single article
2. Batch summarisation of all articles in a single SQL statement
3. Varying summary length by adjusting the prompt approach
4. Domain-specific summarisation with `CORTEX.COMPLETE` for comparison

In [ ]:
import os
import pandas as pd
import snowflake.connector

conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "SFCOGSOPS-SNOWHOUSE_AWS_US_WEST_2"
)
conn.cursor().execute("USE DATABASE GENAI_LEARNING")
conn.cursor().execute("USE SCHEMA PUBLIC")
print("Connected:", conn.account)

## Step 2 — Data Exploration

In [ ]:
articles_df = pd.read_sql("SELECT article_id, title, category, word_count FROM wiki_articles ORDER BY article_id", conn)
print("Articles loaded:")
articles_df

## Step 3 — Summarise a Single Article

`AI_SUMMARIZE(text)` returns a concise summary. No parameters beyond the text — the function determines appropriate length automatically.

In [ ]:
single = pd.read_sql("""
    SELECT title, word_count, AI_SUMMARIZE(body) AS summary
    FROM wiki_articles
    WHERE title = 'Large language model'
""", conn)
print(f"Original: {single['WORD_COUNT'][0]} words")
print(f"Summary: ~{len(single['SUMMARY'][0].split())} words")
print(f"\n{single['SUMMARY'][0]}")

## Step 4 — Batch Summarise All Articles

In [ ]:
batch_df = pd.read_sql("""
    SELECT
        title,
        category,
        word_count,
        AI_SUMMARIZE(body) AS summary
    FROM wiki_articles
    ORDER BY article_id
""", conn)

for _, row in batch_df.iterrows():
    print(f"\n{'='*60}")
    print(f"Title: {row['TITLE']} ({row['WORD_COUNT']} words)")
    print(f"Summary: {row['SUMMARY']}")

## Step 5 — Targeted Summarisation with CORTEX.COMPLETE

`AI_SUMMARIZE` produces a general summary. Use `CORTEX.COMPLETE` when you need a summary focused on a specific angle — e.g. "what are the competitive implications?" or "what are the risks?"

In [ ]:
targeted_df = pd.read_sql("""
    SELECT
        title,
        SNOWFLAKE.CORTEX.COMPLETE(
            'claude-3-5-haiku',
            'Summarise this article in 2 bullet points focused on the key technical capabilities and business use cases. Format as: "• [point]"\n\n' || body
        ) AS technical_summary
    FROM wiki_articles
    ORDER BY article_id
""", conn)

for _, row in targeted_df.iterrows():
    print(f"\n{row['TITLE']}")
    print(row["TECHNICAL_SUMMARY"])

## Step 6 — Interpretation & SA Tips

**AI_SUMMARIZE vs. CORTEX.COMPLETE for summarisation:**

| Use | Function |
|---|---|
| General-purpose summary | `AI_SUMMARIZE` (simpler, no prompt needed) |
| Summary focused on a specific angle | `CORTEX.COMPLETE` with instruction |
| Bullet-point or structured summary | `CORTEX.COMPLETE` |

**SA tips:**
- `AI_SUMMARIZE` works on text up to the model's context window — for very long documents, consider chunking first
- Materialise summaries into a table with `CREATE TABLE AS SELECT` to avoid re-running the LLM on every query
- Combine with Cortex Search: store summaries as the searchable text for faster, cheaper RAG retrieval
- Use `AI_SUMMARIZE` in a Dynamic Table to keep summaries fresh as source documents are updated